# Relatório Final - Data Science
## Bacharelado em Ciência da Computação / PUCPR (2026-1)

**Prof. Rayson Laroca**

Alander Menezes Arantes de Ávila - menezes.alander@pucpr.edu.br

Giancarlo Nunes Perli - giancarlo.perli@sanrocco.com.br

Gustavo Faria Cardoso - faria.cardoso@pucpr.edu.br

Paulo Henrique Perin - paulo.perin@pucpr.edu.br

Pedro Lucas Ghezzi Bittencourt - pedro.bittencourt@pucpr.edu.br

# Importações

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew as _skew, kurtosis as _kurt
from matplotlib.patches import Ellipse as MplEllipse, Patch
from matplotlib.lines import Line2D

#### O dataset foi adquirido através do Kaggle, é mantido pelo Laboratório de Propulsão a Jato do Instituto de Tecnologia da California, uma organização sob a NASA, e contém os seguintes atributos:

**Identificadores** — `id`, `spkid`, `full_name`, `pdes`, `name`, `prefix`, `orbit_id`: identificam entradas no catálogo, sem valor preditivo.

**Classificação**
- `neo` — *Near-Earth Object*: indica se o asteroide orbita na região próxima à Terra (Y/N).
- `pha` — *Potentially Hazardous Asteroid*: indica se o asteroide é classificado como potencialmente perigoso (Y/N). ⭐ **Variável-alvo do projeto.**

**Características físicas**
- `H` — Magnitude absoluta: medida de brilho intrínseco do asteroide. Quanto menor o valor, mais brilhante e geralmente maior o objeto.
- `diameter` — Diâmetro físico em km.
- `albedo` — Fração da luz solar refletida pela superfície (0 = completamente escuro, 1 = completamente reflexivo).

**Parâmetros orbitais**
- `e` — Excentricidade: descreve o formato da órbita. 0 = circular, próximo de 1 = muito alongada.
- `a` — Semi-eixo maior (AU): distância média do asteroide ao Sol.
- `q` — Periélio (AU): ponto mais próximo do Sol na órbita.
- `ad` — Afélio (AU): ponto mais distante do Sol na órbita.
- `i` — Inclinação (°): ângulo entre o plano da órbita do asteroide e o plano da eclíptica (plano da órbita da Terra).
- `w` — Argumento do periélio (°): orientação do ponto mais próximo ao Sol dentro do plano orbital.
- `n` — Movimento médio (°/dia): velocidade angular média ao longo da órbita.
- `per_y` — Período orbital em anos: tempo para completar uma volta ao redor do Sol.
- `moid` — Distância mínima de interseção orbital (AU): menor distância geométrica possível entre a órbita do asteroide e a da Terra.
- `moid_ld` — Mesmo que `moid`, expresso em distâncias lunares.

**Qualidade do ajuste orbital**
- `sigma_e` — Incerteza formal na excentricidade estimada.
- `rms` — Resíduo do ajuste orbital (arcsec): indica a qualidade do ajuste da órbita calculada às observações reais.

**Outros** — `class`: classe orbital dinâmica (MBA, APO, ATE, AMO, IEO…)

## **Objetivos do Projeto**

**Objetivo principal:** Construir um modelo de Machine Learning capaz de classificar, a partir de características físicas e/ou orbitais, se um asteroide é potencialmente perigoso (`pha = Y`), de forma que seja capaz de generalizar para asteroides não classificados.

**Objetivos secundários:**
- Identificar quais características físicas e orbitais melhor diferenciam PHAs dos demais asteroides.
- Avaliar a viabilidade das variáveis físicas (`diameter`, `albedo`, `H`) como features, considerando a alta taxa de valores ausentes.
- Investigar o desbalanceamento de classes e suas implicações para a modelagem.
- Comparar modelos com e sem `moid_ld`, avaliando se os parâmetros orbitais restantes são suficientes para a classificação.

## **Hipóteses**

- **H1:** Asteroides próximos a Terra tendem a ser mais perigosos.
- **H2:** Asteroides com maiores dimensões físicas são mais perigosos.
- **H3:** A geometria da órbita determina se um asteroide cruza a região da Terra.

# Carregamento dos Dados

In [3]:
ds = pd.read_csv(
    "https://github.com/aland3r/asteroides/releases/download/dataset/asteroids-dataset.csv",
    low_memory=False
)
print(f"Shape original: {ds.shape}")

Shape original: (958524, 45)


# Limpeza e Tratamento

O dataset completo possui 958.524 registros e 45 colunas. O primeiro passo da limpeza foi remover as colunas identificadoras (`id`, `spkid`, `full_name`, `pdes`, `name`, `prefix` e `orbit_id`), que não carregam informação analítica. Em seguida, removemos os registros sem rótulo definido (`pha = NaN`), já que a variável alvo é indispensável para as análises supervisionadas.

Em seguida, removemos as colunas `diameter`, `albedo`, `diameter_sigma` e `diameter_bc`, todas com mais de 85% de valores ausentes — imputar essa quantidade seria essencialmente inventar dados. Antes de removê-las, verificamos se os valores presentes estavam concentrados nos PHAs, o que indicaria relevância preditiva. Não era o caso: a grande maioria dos valores pertencia à classe negativa (135.988 não-PHAs contra 221 PHAs para `diameter`), confirmando que a remoção não compromete a identificação de asteroides perigosos.

Por fim, filtramos o dataset para conter apenas NEOs (`neo = Y`). Como todo PHA é necessariamente um objeto próximo à Terra, os asteroides com `neo = N` nunca poderiam ser classificados como perigosos — sua inclusão apenas adicionaria ruído ao problema. Esse filtro reduz o dataset de 938.603 para 22.895 instâncias

In [58]:
ds.isnull().sum()

id                     0
spkid                  0
full_name              0
pdes                   0
name              936460
prefix            958506
neo                    4
pha                19921
H                   6263
diameter          822315
albedo            823421
diameter_sigma    822443
orbit_id               0
epoch                  0
epoch_mjd              0
epoch_cal              0
equinox                0
e                      0
a                      0
q                      0
i                      0
om                     0
w                      0
ma                     1
ad                     4
n                      0
tp                     0
tp_cal                 0
per                    4
per_y                  1
moid               19921
moid_ld              127
sigma_e            19922
sigma_a            19922
sigma_q            19922
sigma_i            19922
sigma_om           19922
sigma_w            19922
sigma_ma           19922
sigma_ad           19926


In [40]:
df_clean = ds.dropna(subset=['pha'])

id_cols = ['id', 'spkid', 'full_name', 'pdes', 'name', 'prefix', 'orbit_id']
high_missing = ['diameter', 'albedo', 'diameter_sigma', 'diameter_bc']
df_clean = df_clean.drop(columns=id_cols + high_missing, errors='ignore')

# Filtrar apenas NEOs — todo PHA é necessariamente um NEO
df_clean = df_clean[df_clean['neo'] == 'Y']
df_clean = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_clean.shape)
print(df_clean['pha'].value_counts())

(22894, 35)
pha
N    20828
Y     2066
Name: count, dtype: int64


### **Amostragem Balanceada**

O desbalanceamento de classes em `pha` dificulta a comparação visual entre os dois grupos. Para contornar isso, construímos `df_equal`: um subconjunto com igual número de PHAs e não-PHAs (2.066 de cada), totalizando 4.132 instâncias, com semente aleatória fixa para reprodutibilidade. Estatísticas populacionais e treinamento dos modelos utilizam `df_clean`; `df_equal` é reservado para visualizações exploratórias onde uma comparação visual justa entre as classes é necessária.

In [37]:
# df_clean contém apenas pha em {'Y','N'}

# separar classes
df_y = df_clean[df_clean['pha'] == 'Y'].copy()
df_n = df_clean[df_clean['pha'] == 'N'].copy()

# amostra balanceada: mesma quantidade de N que de Y
df_n_bal = df_n.sample(n=len(df_y), random_state=42)

# conjunto balanceado final
df_equal = pd.concat([df_y, df_n_bal], ignore_index=True)
df_equal = df_equal.sample(frac=1, random_state=42).reset_index(drop=True)

# verificação
print("df_equal shape:", df_equal.shape)
print(df_equal['pha'].value_counts())
print(df_equal['pha'].value_counts(normalize=True))

df_equal shape: (4132, 35)
pha
Y    2066
N    2066
Name: count, dtype: int64
pha
Y    0.5
N    0.5
Name: proportion, dtype: float64


---

# Descrição Estatística



Após a limpeza:

- 43.386 instâncias (2.066 PHAs e 41.320 não-PHAs);
- 38 atributos;
- **Problema:** classificação binária — `pha = Y` ou `pha = N`
- **Desbalanceamento:** mesmo após a redução, apenas ~4,8% dos asteroides são PHAs — qualquer modelo que preveja sempre "não perigoso" acertaria ~95% das vezes, tornando a acurácia uma métrica inadequada

In [ ]:
# Cores das classes
PHA_COLORS = {
    'Y': '#FF8C00',  # darkorange - Potentially Hazardous
    'N': '#4682B4'   # steelblue - Non-Hazardous
}